<a href="https://colab.research.google.com/github/aymuos/endgame/blob/main/02_batch_level_feature_creator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Batch level feature aggregation

## Why batch aggregation :

 Once a courier receives a batch, every subsequent delivery within it alters their physical state — location, fatigue, remaining time which changes the cost of every remaining delivery. Kool et al. (2019, "Attention, Learn to Solve Routing Problems!") formalise this as a combinatorial dependency: the value of delivering order i depends on what orders have already been served.

 Within a batch, all confounders are held constant: **same courier**, ***same clock time**, **same workload_causal**, **same WSI**, **same hour_sin/cos**, same spatial_congestion_daily (approximately — congestion varies by POI location but the hour is fixed)**. The only source of within-batch variation in eta_mins is the sequencing of orders and individual order characteristics (distance, typecode).  

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
path = '/content/drive/MyDrive/ml/CORRECTEDv3/batchaggregated_all_cities.parquet'

In [4]:
import pandas as pd
df = pd.read_parquet(path)
print(f"DataFrame loaded with {len(df)} rows and {len(df.columns)} columns.")

DataFrame loaded with 40156 rows and 15 columns.


Family A — Spatial Structure (from POI coordinates)

In [5]:
pip install polars

In [6]:
import polars as pl

# Convert pandas DataFrame to polars DataFrame
pl_df = pl.from_pandas(df)
print("Converted pandas DataFrame to polars DataFrame.")

# Display the head of the polars DataFrame to verify
print(pl_df.head())

Converted pandas DataFrame to polars DataFrame.
shape: (5, 15)
┌────────────┬────────────┬───────────┬───────────┬───┬──────────┬───────────┬───────────┬─────────┐
│ batch_id   ┆ workload_c ┆ pickup_de ┆ speed_mea ┆ … ┆ hour_sin ┆ is_holida ┆ is_weeken ┆ eta_max │
│ ---        ┆ ausal      ┆ stination ┆ n         ┆   ┆ ---      ┆ y         ┆ d         ┆ ---     │
│ str        ┆ ---        ┆ _distance ┆ ---       ┆   ┆ f64      ┆ ---       ┆ ---       ┆ f64     │
│            ┆ f64        ┆ ---       ┆ f64       ┆   ┆          ┆ i8        ┆ i8        ┆         │
│            ┆            ┆ f64       ┆           ┆   ┆          ┆           ┆           ┆         │
╞════════════╪════════════╪═══════════╪═══════════╪═══╪══════════╪═══════════╪═══════════╪═════════╡
│ 0008c2b6a2 ┆ 0.0        ┆ 1247.3159 ┆ 0.0       ┆ … ┆ 0.866025 ┆ 0         ┆ 0         ┆ 211.0   │
│ 314db87153 ┆            ┆ 38        ┆           ┆   ┆          ┆           ┆           ┆         │
│ 01b7eeeebc ┆            ┆ 

The `add_batch_spatial_features` function calculates spatial spread and courier-to-centroid distance for each batch, as described in Daganzo's zone theory. These features quantify the geographical characteristics of deliveries within a batch.

In [7]:
def add_batch_spatial_features(features: pl.DataFrame) -> pl.DataFrame:
    batch_geo = (
        features
        .group_by("batch_id")
        .agg([
            pl.col("poi_lat").std().fill_null(0).alias("lat_std"),
            pl.col("poi_lng").std().fill_null(0).alias("lng_std"),
            pl.col("poi_lat").mean().alias("centroid_lat"),
            pl.col("poi_lng").mean().alias("centroid_lng"),
            pl.col("courier_x").first().alias("courier_x"),
            pl.col("courier_y").first().alias("courier_y"),
        ])
        .with_columns([
            # A1: Spatial spread — Daganzo (1984) zone width proxy
            ((pl.col("lat_std").pow(2) + pl.col("lng_std").pow(2)).sqrt())
            .alias("batch_spatial_spread"),

            # A2: Courier-to-centroid distance — access cost to batch
            ((pl.col("courier_x") - pl.col("centroid_lat")).pow(2) +
             (pl.col("courier_y") - pl.col("centroid_lng")).pow(2)).sqrt()
            .alias("batch_centroid_access"),
        ])
        .select(["batch_id", "batch_spatial_spread",
                 "batch_centroid_access",
                 "centroid_lat", "centroid_lng"])
    )
    return features.join(batch_geo, on="batch_id", how="left")

In [8]:
# Apply the function to the polars DataFrame
pl_df_with_spatial_features = add_batch_spatial_features(pl_df)

# Display the head of the DataFrame with new features
print(pl_df_with_spatial_features.head())

ColumnNotFoundError: unable to find column "poi_lat"; valid columns: ["batch_id", "workload_causal", "pickup_destination_distance", "speed_mean", "spatial_congestion_norm", "courier_local_load", "delivery_user_id", "batch_size", "city", "receipt_time", "WSI", "hour_sin", "is_holiday", "is_weekend", "eta_max"]

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
DF ["batch_id", "workload_causal", "pickup_destination_distance", "speed_mean", ...]; PROJECT */15 COLUMNS

In [ ]:
# Optionally, convert the polars DataFrame back to pandas if needed
df = pl_df_with_spatial_features.to_pandas()
print("Converted polars DataFrame back to pandas DataFrame.")
print(df.head())